<a href="https://colab.research.google.com/github/KarlTPs/5IM1_ABTLN_ProyectoFinal/blob/main/5IM1_ABTLN_ProyectoFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import argparse
import json
import math
import random
import re
import time
import unicodedata
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn

PAD, UNK, BOS, EOS = 0, 1, 2, 3

NAH_BASE = ("https://raw.githubusercontent.com/AmericasNLP/americasnlp2021/"
            "main/data/nahuatl-spanish/")
EUROPARL_URL = "https://object.pouta.csc.fi/OPUS-Europarl/v8/moses/es-sv.txt.zip"

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# descarga y lectura de datos

def download(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        print(f"  descargando {url}")
        urllib.request.urlretrieve(url, dest)
    return dest


def read_lines(path_or_text) -> list:
    text = Path(path_or_text).read_text(encoding="utf-8") if isinstance(path_or_text, (str, Path)) else path_or_text
    lines = text.split("\n")  # no usar splitlines(): parte en U+2028 y rompe la alineación
    if lines and lines[-1] == "":
        lines.pop()
    return lines


def load_nahuatl(data_dir: Path):
    es = read_lines(download(NAH_BASE + "train.es", data_dir / "nah" / "train.es"))
    nah = read_lines(download(NAH_BASE + "train.nah", data_dir / "nah" / "train.nah"))
    assert len(es) == len(nah), "El corpus ES-NAH no está alineado"
    return es, nah


def load_swedish(data_dir: Path, max_raw: int, seed: int):
    zpath = download(EUROPARL_URL, data_dir / "sv" / "es-sv.txt.zip")
    with zipfile.ZipFile(zpath) as z:
        names = z.namelist()
        f_es = next(n for n in names if n.endswith(".es"))
        f_sv = next(n for n in names if n.endswith(".sv"))
        es = read_lines(z.read(f_es).decode("utf-8"))
        sv = read_lines(z.read(f_sv).decode("utf-8"))
    assert len(es) == len(sv), "El corpus ES-SV no está alineado"
    if max_raw and len(es) > max_raw:  # muestreo aleatorio (Europarl está ordenado cronológicamente)
        idx = sorted(random.Random(seed).sample(range(len(es)), max_raw))
        es, sv = [es[i] for i in idx], [sv[i] for i in idx]
    return es, sv

In [ ]:
# limpieza
_CTRL = re.compile(r"[\x00-\x08\x0b-\x1f\x7f]")
_ALPHA = re.compile(r"[^\W\d_]")


def normalize(s):
    if not isinstance(s, str):
        return np.nan
    s = unicodedata.normalize("NFC", s)
    s = _CTRL.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else np.nan


def clean_pairs(src, tgt, max_words=60, max_ratio=3.0):
    """Devuelve (DataFrame limpio, dict con el conteo tras cada filtro)."""
    df = pd.DataFrame({"src": src, "tgt": tgt})
    stats = {"crudo": len(df)}
    df["src"] = df["src"].map(normalize)
    df["tgt"] = df["tgt"].map(normalize)
    df = df.dropna()
    stats["sin_nulos_ni_vacios"] = len(df)
    df = df.drop_duplicates()
    stats["sin_pares_duplicados"] = len(df)
    df = df[~df["src"].str.lower().duplicated()]  # evita fuga train/test por fuentes repetidas
    stats["sin_fuentes_duplicadas"] = len(df)
    df = df[df["src"] != df["tgt"]]  # "traducciones" que son copia
    df = df[df["src"].map(lambda s: bool(_ALPHA.search(s))) & df["tgt"].map(lambda s: bool(_ALPHA.search(s)))]
    stats["sin_copias_ni_sin_letras"] = len(df)
    ns, nt = df["src"].str.split().str.len(), df["tgt"].str.split().str.len()
    keep = (ns <= max_words) & (nt <= max_words) & (ns / nt <= max_ratio) & (nt / ns <= max_ratio)
    df = df[keep].reset_index(drop=True)
    stats["tras_filtro_longitud"] = len(df)
    return df, stats


def split_df(df, seed, frac=(0.8, 0.1, 0.1)):
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    a, b = int(n * frac[0]), int(n * (frac[0] + frac[1]))
    return df.iloc[:a].reset_index(drop=True), df.iloc[a:b].reset_index(drop=True), df.iloc[b:].reset_index(drop=True)

In [ ]:
# tokenización

def train_tokenizer(train_df, prefix: Path, vocab_size: int):
    prefix.parent.mkdir(parents=True, exist_ok=True)
    txt = f"{prefix}.train.txt"
    with open(txt, "w", encoding="utf-8") as f:
        for s in list(train_df["src"]) + list(train_df["tgt"]):
            f.write(s + "\n")
    spm.SentencePieceTrainer.train(
        f"--input={txt} --model_prefix={prefix} --vocab_size={vocab_size} --model_type=bpe "
        f"--character_coverage=1.0 --pad_id={PAD} --unk_id={UNK} --bos_id={BOS} --eos_id={EOS} "
        f"--hard_vocab_limit=false --minloglevel=2")
    return spm.SentencePieceProcessor(model_file=f"{prefix}.model")


def encode_split(sp, df, max_len):
    """Lista de (ids_src, ids_tgt, texto_src, texto_tgt). Descarta secuencias > max_len."""
    out = []
    for s, t in zip(df["src"], df["tgt"]):
        a = sp.encode(s) + [EOS]
        b = [BOS] + sp.encode(t) + [EOS]
        if len(a) <= max_len and len(b) <= max_len:
            out.append((a, b, s, t))
    return out


def pad_batch(seqs):
    L = max(len(s) for s in seqs)
    return torch.tensor([s + [PAD] * (L - len(s)) for s in seqs], dtype=torch.long)


def make_batches(data, bs, shuffle, rng):
    idx = list(range(len(data)))
    if shuffle:  # agrupa por longitud dentro de bloques para reducir padding
        rng.shuffle(idx)
        blocks = [idx[i:i + bs * 50] for i in range(0, len(idx), bs * 50)]
        idx = [j for blk in blocks for j in sorted(blk, key=lambda k: len(data[k][0]))]
    else:
        idx = sorted(idx, key=lambda k: len(data[k][0]))
    batches = [idx[i:i + bs] for i in range(0, len(idx), bs)]
    if shuffle:
        rng.shuffle(batches)
    return batches

In [ ]:
# modelo: transformer encoder-decoder, pesos aleatorios

class Seq2Seq(nn.Module):
    def __init__(self, vocab, d_model, n_heads, n_layers, ff_mult, dropout, max_pos=512):
        super().__init__()
        self.d_model = d_model
        self.emb = nn.Embedding(vocab, d_model, padding_idx=PAD)
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_pos, d_model)
        pos = torch.arange(max_pos).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)
        self.tf = nn.Transformer(d_model, n_heads, n_layers, n_layers, d_model * ff_mult,
                                 dropout, batch_first=True, norm_first=True)
        self.out = nn.Linear(d_model, vocab, bias=False)
        for p in self.tf.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        nn.init.normal_(self.emb.weight, 0.0, d_model ** -0.5)
        with torch.no_grad():
            self.emb.weight[PAD].zero_()
        self.out.weight = self.emb.weight  # embeddings compartidos (vocabulario conjunto)

    def _embed(self, x):
        return self.drop(self.emb(x) * math.sqrt(self.d_model) + self.pe[: x.size(1)])

    def encode(self, src):
        mask = src.eq(PAD)
        return self.tf.encoder(self._embed(src), src_key_padding_mask=mask), mask

    def decode(self, tgt_in, memory, mem_mask):
        T = tgt_in.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt_in.device), 1)
        h = self.tf.decoder(self._embed(tgt_in), memory, tgt_mask=causal,
                            memory_key_padding_mask=mem_mask)
        return self.out(h)

    def forward(self, src, tgt_in):
        mem, mm = self.encode(src)
        return self.decode(tgt_in, mem, mm)


def build_model(cfg, vocab):
    return Seq2Seq(vocab, cfg["d_model"], cfg["n_heads"], cfg["n_layers"],
                   cfg["ff_mult"], cfg["dropout"])

In [ ]:
# entrenamiento, decodificación y métricas

@torch.no_grad()
def eval_loss(model, data, vocab, device, bs=64):
    model.eval()
    crit = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")
    total, ntok = 0.0, 0
    for b in make_batches(data, bs, False, None):
        src = pad_batch([data[i][0] for i in b]).to(device)
        tgt = pad_batch([data[i][1] for i in b]).to(device)
        logits = model(src, tgt[:, :-1])
        total += crit(logits.reshape(-1, vocab), tgt[:, 1:].reshape(-1)).item()
        ntok += tgt[:, 1:].ne(PAD).sum().item()
    return total / max(ntok, 1)


def train_model(cfg, train, val, vocab, epochs, device, seed, patience=3, verbose=False):
    set_seed(seed)
    model = build_model(cfg, vocab).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], betas=(0.9, 0.98), eps=1e-9)
    total_steps = math.ceil(len(train) / cfg["batch_size"]) * epochs
    warm = max(30, int(0.1 * total_steps))  # calentamiento + decaimiento inverso a la raíz
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, math.sqrt(warm / (s + 1))))
    crit = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=cfg["label_smoothing"])
    rng = random.Random(seed)
    best, best_state, bad, hist = float("inf"), None, 0, []
    for ep in range(1, epochs + 1):
        model.train()
        run, nb = 0.0, 0
        for b in make_batches(train, cfg["batch_size"], True, rng):
            src = pad_batch([train[i][0] for i in b]).to(device)
            tgt = pad_batch([train[i][1] for i in b]).to(device)
            logits = model(src, tgt[:, :-1])
            loss = crit(logits.reshape(-1, vocab), tgt[:, 1:].reshape(-1))
            if not torch.isfinite(loss):
                return model, float("inf"), hist  # configuración inestable
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            run, nb = run + loss.item(), nb + 1
        vl = eval_loss(model, val, vocab, device)
        hist.append({"epoch": ep, "train_loss": run / max(nb, 1), "val_loss": vl})
        if verbose:
            print(f"    época {ep:02d}  train {run / max(nb, 1):.3f}  val {vl:.3f}")
        if vl < best - 1e-4:
            best, bad = vl, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best, hist


@torch.no_grad()
def translate(model, srcs, device, bs=64):
    """Decodificación voraz (greedy). Devuelve listas de ids sin BOS/EOS."""
    model.eval()
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    out = [None] * len(srcs)
    for s in range(0, len(order), bs):
        ids = order[s:s + bs]
        src = pad_batch([srcs[i] for i in ids]).to(device)
        mem, mm = model.encode(src)
        ys = torch.full((len(ids), 1), BOS, dtype=torch.long, device=device)
        done = torch.zeros(len(ids), dtype=torch.bool, device=device)
        for _ in range(src.size(1) * 2 + 10):
            nxt = model.decode(ys, mem, mm)[:, -1].argmax(-1).masked_fill(done, PAD)
            ys = torch.cat([ys, nxt[:, None]], 1)
            done |= nxt.eq(EOS)
            if done.all():
                break
        for k, i in enumerate(ids):
            seq = []
            for t in ys[k, 1:].tolist():
                if t in (EOS, PAD):
                    break
                seq.append(t)
            out[i] = seq
    return out


_TOK = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def toks(s):
    return _TOK.findall(s.lower())


def prf1(hyps, refs):
    """Precisión, exhaustividad y F1 a nivel de token (conteos recortados, micro-promedio)."""
    tp = hn = rn = 0
    for h, r in zip(hyps, refs):
        ch, cr = Counter(toks(h)), Counter(toks(r))
        tp += sum((ch & cr).values())
        hn += sum(ch.values())
        rn += sum(cr.values())
    p = tp / hn if hn else 0.0
    r = tp / rn if rn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def corpus_bleu(hyps, refs, max_n=4):
    match, total, hl, rl = [0] * max_n, [0] * max_n, 0, 0
    for h, r in zip(hyps, refs):
        h, r = toks(h), toks(r)
        hl, rl = hl + len(h), rl + len(r)
        for n in range(1, max_n + 1):
            hc = Counter(tuple(h[i:i + n]) for i in range(len(h) - n + 1))
            rc = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
            match[n - 1] += sum(min(c, rc[g]) for g, c in hc.items())
            total[n - 1] += max(len(h) - n + 1, 0)
    if hl == 0 or total[0] == 0:
        return 0.0
    logp = 0.0
    for n in range(max_n):
        sm = 0 if n == 0 else 1  # suavizado add-1 para n>1
        p = (match[n] + sm) / (total[n] + sm) if total[n] + sm else 0.0
        if p == 0:
            return 0.0
        logp += math.log(p) / max_n
    bp = 1.0 if hl > rl else math.exp(1 - rl / hl)
    return 100.0 * bp * math.exp(logp)


def evaluate(model, sp, data, device):
    hyp_ids = translate(model, [d[0] for d in data], device)
    hyps = [sp.decode(h) for h in hyp_ids]
    refs = [d[3] for d in data]
    p, r, f = prf1(hyps, refs)
    return {"precision": p, "recall": r, "f1": f, "bleu": corpus_bleu(hyps, refs)}, hyps

In [ ]:
# murcielagos para hiperparametros

# cada murciélago es un vector en [0,1]^D que se decodifica a una configuración
SPACE = [
    ("d_model",         "choice", [128, 256, 384]),
    ("n_heads",         "choice", [2, 4, 8]),
    ("n_layers",        "choice", [1, 2, 3, 4]),
    ("ff_mult",         "choice", [2, 4]),
    ("dropout",         "float",  (0.05, 0.40)),
    ("lr",              "log",    (2e-4, 2e-3)),
    ("label_smoothing", "float",  (0.0, 0.20)),
    ("batch_size",      "choice", [32, 64, 128]),
]

# configuración base (típica de Transformers pequeños para bajos recursos)
BASELINE = dict(d_model=256, n_heads=4, n_layers=3, ff_mult=4, dropout=0.1,
                lr=5e-4, label_smoothing=0.1, batch_size=64)


def decode_vec(x):
    cfg = {}
    for xi, (name, kind, rng) in zip(x, SPACE):
        xi = float(min(max(xi, 0.0), 1.0))
        if kind == "choice":
            cfg[name] = rng[min(int(xi * len(rng)), len(rng) - 1)]
        elif kind == "float":
            cfg[name] = round(rng[0] + xi * (rng[1] - rng[0]), 4)
        else:
            cfg[name] = float(f"{math.exp(math.log(rng[0]) + xi * (math.log(rng[1]) - math.log(rng[0]))):.4g}")
    return cfg


def encode_cfg(cfg):
    v = []
    for name, kind, rng in SPACE:
        if kind == "choice":
            v.append((rng.index(cfg[name]) + 0.5) / len(rng))
        elif kind == "float":
            v.append((cfg[name] - rng[0]) / (rng[1] - rng[0]))
        else:
            v.append((math.log(cfg[name]) - math.log(rng[0])) / (math.log(rng[1]) - math.log(rng[0])))
    return np.array(v)


def bat_algorithm(fitness, n_bats, n_iter, seed, init_vectors=(), fmin=0.0, fmax=2.0,
                  alpha=0.9, gamma=0.9, local_scale=0.1, log=print):
    """
    Maximiza `fitness(cfg)`.
      f_i  = fmin + (fmax-fmin)*beta           frecuencia
      v_i += (x* - x_i) * f_i ; x_i += v_i     (se usa x* - x_i para que el movimiento sea hacia el mejor)
      búsqueda local si rand > r_i:  x = x* + local_scale * eps * media(A)
      se acepta si rand < A_i y mejora; entonces  A_i *= alpha ;  r_i = r_i0 * (1 - exp(-gamma t))
    """
    rng = np.random.default_rng(seed)
    D = len(SPACE)
    X = rng.random((n_bats, D))
    for i, v in enumerate(init_vectors):
        X[i] = v
    Vel = np.zeros_like(X)
    A = rng.uniform(1.0, 2.0, n_bats)
    r0 = rng.uniform(0.0, 1.0, n_bats)
    r = r0.copy()

    fit = np.zeros(n_bats)
    for i in range(n_bats):
        fit[i] = fitness(decode_vec(X[i]))
    b = int(np.argmax(fit))
    xbest, fbest = X[b].copy(), float(fit[b])
    history = [{"iter": 0, "best_f1": fbest, "best_cfg": decode_vec(xbest)}]
    log(f"  [BAT] iter 0  mejor F1(val)={fbest:.4f}")

    for t in range(1, n_iter + 1):
        for i in range(n_bats):
            f = fmin + (fmax - fmin) * rng.random()
            Vel[i] += (xbest - X[i]) * f
            xn = np.clip(X[i] + Vel[i], 0.0, 1.0)
            if rng.random() > r[i]:  # caminata local alrededor del mejor
                xn = np.clip(xbest + local_scale * rng.uniform(-1, 1, D) * A.mean(), 0.0, 1.0)
            fn = fitness(decode_vec(xn))
            if fn > fit[i] and rng.random() < A[i]:
                X[i], fit[i] = xn, fn
                A[i] *= alpha
                r[i] = r0[i] * (1 - math.exp(-gamma * t))
            if fn > fbest:
                xbest, fbest = xn.copy(), float(fn)
        history.append({"iter": t, "best_f1": fbest, "best_cfg": decode_vec(xbest)})
        log(f"  [BAT] iter {t}  mejor F1(val)={fbest:.4f}")
    return decode_vec(xbest), fbest, history




In [ ]:
# pipeline por par de idiomas

def run_pair(name, src, tgt, args, device):
    out = Path(args.out_dir) / name
    out.mkdir(parents=True, exist_ok=True)
    print(f"\n{'=' * 70}\n  ES -> {name.upper()}\n{'=' * 70}")

    # limpieza y partición
    df, stats = clean_pairs(src, tgt, max_words=args.max_words)
    print("  Limpieza:", json.dumps(stats, ensure_ascii=False))
    tr_df, va_df, te_df = split_df(df, args.seed)
    print(f"  Split: train={len(tr_df)} val={len(va_df)} test={len(te_df)}")

    # tokenización
    vocab_size = args.vocab_nah if name == "nah" else args.vocab_sv
    sp = train_tokenizer(tr_df, out / "spm", vocab_size)
    V = sp.get_piece_size()
    train, val, test = (encode_split(sp, d, args.max_len) for d in (tr_df, va_df, te_df))
    print(f"  Vocab={V}  secuencias (<= {args.max_len} subpalabras): "
          f"train={len(train)} val={len(val)} test={len(test)}")

    results = {"limpieza": stats, "vocab": V,
               "n_train": len(train), "n_val": len(val), "n_test": len(test)}

    # modelo base
    print("\n  >> Entrenando modelo BASE", BASELINE)
    t0 = time.time()
    model, vl, hist = train_model(BASELINE, train, val, V, args.epochs, device, args.seed, args.patience, verbose=True)
    m_base, hyp_base = evaluate(model, sp, test, device)
    results["base"] = {"cfg": BASELINE, "val_loss": vl, "test": m_base, "historia": hist,
                       "segundos": round(time.time() - t0, 1)}
    print(f"  BASE  test: P={m_base['precision']:.4f} R={m_base['recall']:.4f} "
          f"F1={m_base['f1']:.4f} BLEU={m_base['bleu']:.2f}")
    torch.save({"cfg": BASELINE, "state": model.state_dict()}, out / "modelo_base.pt")

    # Bat Algorithm
    rs = random.Random(args.seed)
    s_train = rs.sample(train, min(args.search_train, len(train)))
    s_val = rs.sample(val, min(args.search_val, len(val)))
    cache, trials = {}, []

    def fitness(cfg):
        key = tuple(sorted(cfg.items()))
        if key in cache:
            return cache[key]
        t1 = time.time()
        try:
            m, vl_, _ = train_model(cfg, s_train, s_val, V, args.epochs_search, device, args.seed, patience=2)
            f1 = evaluate(m, sp, s_val, device)[0]["f1"] if math.isfinite(vl_) else 0.0
            del m
        except RuntimeError as e:  # p. ej. memoria insuficiente
            print("    (config descartada:", str(e)[:60], ")")
            f1 = 0.0
        if device.type == "cuda":
            torch.cuda.empty_cache()
        cache[key] = f1
        trials.append({"cfg": cfg, "f1_val": f1})
        print(f"    prueba #{len(trials):02d} F1(val)={f1:.4f}  {cfg}  ({time.time() - t1:.0f}s)")
        return f1

    print(f"\n  >> Bat Algorithm: {args.bats} murciélagos x {args.iters} iteraciones")
    t0 = time.time()
    best_cfg, best_f, bat_hist = bat_algorithm(
        fitness, args.bats, args.iters, args.seed, init_vectors=[encode_cfg(BASELINE)])
    print("  Mejor configuración:", best_cfg)

    # reentrenamiento final con la mejor configuración (mismos datos y épocas que el base)
    print("\n  >> Reentrenando con la configuración optimizada")
    model, vl, hist = train_model(best_cfg, train, val, V, args.epochs, device, args.seed, args.patience, verbose=True)
    m_bat, hyp_bat = evaluate(model, sp, test, device)
    results["bat"] = {"cfg": best_cfg, "fitness_busqueda": best_f, "val_loss": vl, "test": m_bat,
                      "historia_entrenamiento": hist, "historia_bat": bat_hist, "pruebas": trials,
                      "segundos": round(time.time() - t0, 1)}
    print(f"  BAT   test: P={m_bat['precision']:.4f} R={m_bat['recall']:.4f} "
          f"F1={m_bat['f1']:.4f} BLEU={m_bat['bleu']:.2f}")
    torch.save({"cfg": best_cfg, "state": model.state_dict()}, out / "modelo_bat.pt")

    # guardar resultados y ejemplos
    with open(out / "resultados.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    with open(out / "ejemplos_test.txt", "w", encoding="utf-8") as f:
        for i, d in enumerate(test[:30]):
            f.write(f"ES  : {d[2]}\nREF : {d[3]}\nBASE: {hyp_base[i]}\nBAT : {hyp_bat[i]}\n\n")
    return results


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--pairs", nargs="+", default=["nah", "sv"], choices=["nah", "sv"])
    ap.add_argument("--data_dir", default="data")
    ap.add_argument("--out_dir", default="salida")
    ap.add_argument("--max_pairs_nah", type=int, default=0, help="0 = todo el corpus")
    ap.add_argument("--max_pairs_sv", type=int, default=60000, help="pares crudos de Europarl a muestrear")
    ap.add_argument("--max_words", type=int, default=60)
    ap.add_argument("--max_len", type=int, default=64, help="máx. subpalabras por secuencia")
    ap.add_argument("--vocab_nah", type=int, default=4000)
    ap.add_argument("--vocab_sv", type=int, default=8000)
    ap.add_argument("--epochs", type=int, default=15, help="épocas del entrenamiento base y final")
    ap.add_argument("--epochs_search", type=int, default=4, help="épocas por evaluación del Bat Algorithm")
    ap.add_argument("--patience", type=int, default=3)
    ap.add_argument("--bats", type=int, default=6)
    ap.add_argument("--iters", type=int, default=5)
    ap.add_argument("--search_train", type=int, default=8000, help="pares de train usados en la búsqueda")
    ap.add_argument("--search_val", type=int, default=400, help="pares de val usados como fitness")
    ap.add_argument("--seed", type=int, default=42)

    # En entornos interactivos de Jupyter/Colab, parseamos solo los argumentos conocidos
    # para evitar errores con los argumentos del kernel (como '-f')
    args, _ = ap.parse_known_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Dispositivo:", device)
    data_dir = Path(args.data_dir)
    all_results = {}

    for pair in args.pairs:
        if pair == "nah":
            src, tgt = load_nahuatl(data_dir)
            if args.max_pairs_nah:
                src, tgt = src[:args.max_pairs_nah], tgt[:args.max_pairs_nah]
        else:
            src, tgt = load_swedish(data_dir, args.max_pairs_sv, args.seed)
        all_results[pair] = run_pair(pair, src, tgt, args, device)

    print(f"\n{'=' * 70}\n  RESUMEN FINAL (conjunto de test)\n{'=' * 70}")
    print(f"{'par':<6}{'modelo':<8}{'Precisión':>11}{'Recall':>9}{'F1':>9}{'BLEU':>8}")
    for pair, res in all_results.items():
        for k, lab in (("base", "base"), ("bat", "bat")):
            m = res[k]["test"]
            print(f"es-{pair:<3}{lab:<8}{m['precision']:>11.4f}{m['recall']:>9.4f}{m['f1']:>9.4f}{m['bleu']:>8.2f}")


if __name__ == "__main__":
    main()

Dispositivo: cuda

  ES -> NAH


KeyboardInterrupt: 

In [ ]:
%pip install -q transformers sentencepiece accelerate

In [ ]:
import gc, json, math, random, time
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup

_requeridos = ["load_nahuatl", "load_swedish", "clean_pairs", "split_df", "train_tokenizer",
               "encode_split", "train_model", "evaluate", "prf1", "corpus_bleu", "set_seed", "spm", "np"]
_faltan = [n for n in _requeridos if n not in globals()]
assert not _faltan, f"Ejecuta primero las celdas 0-8 del cuaderno original. Faltan: {_faltan}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8  # Ampere o superior
print("Dispositivo:", device, "| bf16:", USE_BF16)
if device.type == "cpu":
    print("AVISO: sin GPU el fine-tuning de un modelo de 600M de parámetros será extremadamente lento.")

Dispositivo: cuda | bf16: False


In [ ]:
def prepare_data(name, max_len=64, seed=42, out_dir="salida", data_dir="data", max_pairs_sv=60000):
    out = Path(out_dir) / name
    if name == "nah":
        src, tgt = load_nahuatl(Path(data_dir))
    else:
        src, tgt = load_swedish(Path(data_dir), max_pairs_sv, seed)
    df, stats = clean_pairs(src, tgt, max_words=60)
    tr_df, va_df, te_df = split_df(df, seed)
    prefix = out / "spm"
    if Path(f"{prefix}.model").exists():          # reutiliza el tokenizador de la corrida original
        sp = spm.SentencePieceProcessor(model_file=f"{prefix}.model")
    else:
        sp = train_tokenizer(tr_df, prefix, 4000 if name == "nah" else 8000)
    train, val, test = (encode_split(sp, d, max_len) for d in (tr_df, va_df, te_df))
    print(f"[{name}] limpieza: {stats}")
    print(f"[{name}] train={len(train)} val={len(val)} test={len(test)}")
    return sp, train, val, test

In [ ]:
BEST_CFG = {'d_model': 256, 'n_heads': 4, 'n_layers': 3, 'ff_mult': 4, 'dropout': 0.06,
            'lr': 0.0006169, 'label_smoothing': 0.0863, 'batch_size': 32}

# Resultado del modelo base (salida del cuaderno original, conjunto de test)
BASE_NAH = {"precision": 0.2926, "recall": 0.3211, "f1": 0.3062, "bleu": 7.43}

sp_nah, train_nah, val_nah, test_nah = prepare_data("nah")
V_nah = sp_nah.get_piece_size()

t0 = time.time()
model_a, vl_a, hist_a = train_model(BEST_CFG, train_nah, val_nah, V_nah, epochs=15,
                                    device=device, seed=42, patience=3, verbose=True)
m_bat_scratch, hyp_a = evaluate(model_a, sp_nah, test_nah, device)
seg_a = round(time.time() - t0, 1)

print(f"\n{'modelo':<28}{'Precisión':>10}{'Recall':>9}{'F1':>9}{'BLEU':>8}")
print(f"{'Scratch base':<28}{BASE_NAH['precision']:>10.4f}{BASE_NAH['recall']:>9.4f}{BASE_NAH['f1']:>9.4f}{BASE_NAH['bleu']:>8.2f}")
print(f"{'Scratch + BAT (mejor cfg)':<28}{m_bat_scratch['precision']:>10.4f}{m_bat_scratch['recall']:>9.4f}"
      f"{m_bat_scratch['f1']:>9.4f}{m_bat_scratch['bleu']:>8.2f}")

out_a = Path("salida/nah"); out_a.mkdir(parents=True, exist_ok=True)
with open(out_a / "resultados_scratch_bat_test.json", "w", encoding="utf-8") as f:
    json.dump({"cfg": BEST_CFG, "test": m_bat_scratch, "val_loss": vl_a, "historia": hist_a,
               "base_test": BASE_NAH, "segundos": seg_a}, f, ensure_ascii=False, indent=2)
with open(out_a / "ejemplos_scratch_bat.txt", "w", encoding="utf-8") as f:
    for i, d in enumerate(test_nah[:30]):
        f.write(f"ES : {d[2]}\nREF: {d[3]}\nBAT: {hyp_a[i]}\n\n")
del model_a
gc.collect()

[nah] limpieza: {'crudo': 16145, 'sin_nulos_ni_vacios': 16062, 'sin_pares_duplicados': 15448, 'sin_fuentes_duplicadas': 15191, 'sin_copias_ni_sin_letras': 15113, 'tras_filtro_longitud': 13069}
[nah] train=9225 val=1146 test=1171


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:145: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


    época 01  train 6.952  val 6.120
    época 02  train 5.740  val 5.266
    época 03  train 4.992  val 4.771
    época 04  train 4.512  val 4.526
    época 05  train 4.165  val 4.334
    época 06  train 3.892  val 4.248
    época 07  train 3.667  val 4.165
    época 08  train 3.474  val 4.113
    época 09  train 3.306  val 4.107
    época 10  train 3.159  val 4.086
    época 11  train 3.030  val 4.076
    época 12  train 2.913  val 4.077
    época 13  train 2.813  val 4.088
    época 14  train 2.718  val 4.090

modelo                       Precisión   Recall       F1    BLEU
Scratch base                    0.2926   0.3211   0.3062    7.43
Scratch + BAT (mejor cfg)       0.3404   0.3488   0.3445    9.57


194

## Parte B. Fine-tuning de un modelo preentrenado con optimización por murciélagos

**Modelo:** `facebook/nllb-200-distilled-600M`, un modelo multilingüe preentrenado (con su propio tokenizador SentencePiece de 256k piezas). Soporta español (`spa_Latn`) y sueco (`swe_Latn`).

**Importante sobre el náhuatl:** hasta donde sé, NLLB-200 no incluye náhuatl entre sus idiomas (verifícalo en la lista oficial). Por eso para ES→NAH se usa una **etiqueta de idioma sustituta** (`quy_Latn`, quechua de Ayacucho). Es solo una etiqueta: el fine-tuning enseña al decodificador a producir náhuatl con ella. Se puede cambiar en `TGT_CODE`. Este punto debe quedar documentado en la investigación como decisión metodológica.

**Espacio de búsqueda del murciélago (7 dimensiones):** `lr` (escala log), `weight_decay`, `warmup_ratio`, `label_smoothing`, `dropout`, `batch_size` y `freeze_embeddings`. Congelar los 262M de parámetros de embeddings ahorra mucha memoria y puede reducir el sobreajuste.

In [ ]:
PRETRAINED = "facebook/nllb-200-distilled-600M"
SRC_CODE = "spa_Latn"
TGT_CODE = {"nah": "quy_Latn",   # etiqueta sustituta: NLLB no trae náhuatl
            "sv":  "swe_Latn"}
MAX_LEN = 64          # tokens por secuencia (el modelo preentrenado usa su propio tokenizador)
EVAL_BEAMS = 4        # búsqueda por haz en la evaluación final; las pruebas del murciélago usan voraz

FT_SPACE = [
    ("lr",                "log",    (1e-5, 3e-4)),
    ("weight_decay",      "float",  (0.0, 0.1)),
    ("warmup_ratio",      "float",  (0.0, 0.2)),
    ("label_smoothing",   "float",  (0.0, 0.2)),
    ("dropout",           "float",  (0.0, 0.3)),
    ("batch_size",        "choice", [8, 16, 32]),
    ("freeze_embeddings", "choice", [True, False]),
]
# Configuración base de fine-tuning (valores habituales)
FT_BASELINE = dict(lr=5e-5, weight_decay=0.01, warmup_ratio=0.1, label_smoothing=0.1,
                   dropout=0.1, batch_size=16, freeze_embeddings=True)


# Codificación / decodificación genérica del vector del murciélago
def decode_space(x, space):
    cfg = {}
    for xi, (name, kind, rng) in zip(x, space):
        xi = float(min(max(xi, 0.0), 1.0))
        if kind == "choice":
            cfg[name] = rng[min(int(xi * len(rng)), len(rng) - 1)]
        elif kind == "float":
            cfg[name] = round(rng[0] + xi * (rng[1] - rng[0]), 4)
        else:  # log
            cfg[name] = float(f"{math.exp(math.log(rng[0]) + xi * (math.log(rng[1]) - math.log(rng[0]))):.4g}")
    return cfg


def encode_space(cfg, space):
    v = []
    for name, kind, rng in space:
        if kind == "choice":
            v.append((rng.index(cfg[name]) + 0.5) / len(rng))
        elif kind == "float":
            v.append((cfg[name] - rng[0]) / (rng[1] - rng[0]))
        else:
            v.append((math.log(cfg[name]) - math.log(rng[0])) / (math.log(rng[1]) - math.log(rng[0])))
    return np.array(v)


# Algoritmo de Murciélagos genérico (misma lógica que el del cuaderno original)
def bat_generic(fitness, space, n_bats, n_iter, seed, init_vectors=(), fmin=0.0, fmax=2.0,
                alpha=0.9, gamma=0.9, local_scale=0.1, log=print):
    rng = np.random.default_rng(seed)
    D = len(space)
    X = rng.random((n_bats, D))
    for i, v in enumerate(init_vectors):
        X[i] = v
    Vel = np.zeros_like(X)
    A = rng.uniform(1.0, 2.0, n_bats)
    r0 = rng.uniform(0.0, 1.0, n_bats)
    r = r0.copy()

    fit = np.array([fitness(decode_space(X[i], space)) for i in range(n_bats)])
    b = int(np.argmax(fit))
    xbest, fbest = X[b].copy(), float(fit[b])
    history = [{"iter": 0, "best_f1": fbest, "best_cfg": decode_space(xbest, space)}]
    log(f"  [BAT] iter 0  mejor F1(val)={fbest:.4f}")
    for t in range(1, n_iter + 1):
        for i in range(n_bats):
            f = fmin + (fmax - fmin) * rng.random()
            Vel[i] += (xbest - X[i]) * f
            xn = np.clip(X[i] + Vel[i], 0.0, 1.0)
            if rng.random() > r[i]:
                xn = np.clip(xbest + local_scale * rng.uniform(-1, 1, D) * A.mean(), 0.0, 1.0)
            fn = fitness(decode_space(xn, space))
            if fn > fit[i] and rng.random() < A[i]:
                X[i], fit[i] = xn, fn
                A[i] *= alpha
                r[i] = r0[i] * (1 - math.exp(-gamma * t))
            if fn > fbest:
                xbest, fbest = xn.copy(), float(fn)
        history.append({"iter": t, "best_f1": fbest, "best_cfg": decode_space(xbest, space)})
        log(f"  [BAT] iter {t}  mejor F1(val)={fbest:.4f}")
    return decode_space(xbest, space), fbest, history

In [ ]:
# Fine-tuning, decodificación y evaluación con el modelo preentrenado
def make_tok(name):
    return AutoTokenizer.from_pretrained(PRETRAINED, src_lang=SRC_CODE, tgt_lang=TGT_CODE[name])


def load_pretrained(dropout=0.1):
    # los pesos vienen preentrenados; solo se ajusta el dropout de la configuración
    return AutoModelForSeq2SeqLM.from_pretrained(PRETRAINED, dropout=dropout).to(device)


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16)


def seq2seq_loss(model, tok, batch_pairs, smoothing):
    enc = tok([p[0] for p in batch_pairs], text_target=[p[1] for p in batch_pairs],
              max_length=MAX_LEN, truncation=True, padding=True, return_tensors="pt")
    labels = enc["labels"].to(device)
    pad = model.config.pad_token_id
    start = model.config.decoder_start_token_id
    dec_in = torch.cat([torch.full((labels.size(0), 1), start, dtype=labels.dtype, device=device),
                        labels[:, :-1]], dim=1)
    target = labels.masked_fill(labels == pad, -100)
    with amp_ctx():
        logits = model(input_ids=enc["input_ids"].to(device),
                       attention_mask=enc["attention_mask"].to(device),
                       decoder_input_ids=dec_in).logits
    return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), target.reshape(-1),
                           ignore_index=-100, label_smoothing=smoothing), target.ne(-100).sum().item()


@torch.no_grad()
def ft_eval_loss(model, tok, pairs, bs=16):
    model.eval()
    total, ntok = 0.0, 0
    for s in range(0, len(pairs), bs):
        loss, n = seq2seq_loss(model, tok, pairs[s:s + bs], 0.0)
        total += loss.item() * n
        ntok += n
    return total / max(ntok, 1)


def finetune(cfg, train_pairs, val_pairs, tok, epochs, seed, verbose=False):
    set_seed(seed)
    model = load_pretrained(cfg["dropout"])
    if cfg["freeze_embeddings"]:
        for p in model.get_input_embeddings().parameters():   # tambien congela la cabeza de salida (pesos atados)
            p.requires_grad = False
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    bs = cfg["batch_size"]
    total_steps = math.ceil(len(train_pairs) / bs) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup_ratio"] * total_steps), total_steps)
    rng = random.Random(seed)
    hist = []
    for ep in range(1, epochs + 1):
        model.train()
        idx = list(range(len(train_pairs)))
        rng.shuffle(idx)
        run, nb = 0.0, 0
        for s in range(0, len(idx), bs):
            batch = [train_pairs[i] for i in idx[s:s + bs]]
            loss, _ = seq2seq_loss(model, tok, batch, cfg["label_smoothing"])
            if not torch.isfinite(loss):
                return model, float("inf"), hist
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            sched.step()
            run, nb = run + loss.item(), nb + 1
        vl = ft_eval_loss(model, tok, val_pairs)
        hist.append({"epoch": ep, "train_loss": run / max(nb, 1), "val_loss": vl})
        if verbose:
            print(f"    época {ep:02d}  train {run / max(nb, 1):.3f}  val {vl:.3f}")
    del opt
    return model, hist[-1]["val_loss"], hist


@torch.no_grad()
def translate_ft(model, tok, name, srcs, num_beams=1, bs=16):
    model.eval()
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    out = [None] * len(srcs)
    forced = tok.convert_tokens_to_ids(TGT_CODE[name])
    for s in range(0, len(order), bs):
        ids = order[s:s + bs]
        enc = tok([srcs[i] for i in ids], return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to(device)
        with amp_ctx():
            gen = model.generate(**enc, forced_bos_token_id=forced, num_beams=num_beams,
                                 max_new_tokens=MAX_LEN + 16)
        for i, h in zip(ids, tok.batch_decode(gen, skip_special_tokens=True)):
            out[i] = h
    return out


def evaluate_ft(model, tok, name, pairs, num_beams=1):
    hyps = translate_ft(model, tok, name, [p[0] for p in pairs], num_beams)
    refs = [p[1] for p in pairs]
    p, r, f = prf1(hyps, refs)
    return {"precision": p, "recall": r, "f1": f, "bleu": corpus_bleu(hyps, refs)}, hyps


def free_gpu():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

In [ ]:
# Pipeline por par de idiomas: base -> Bat Algorithm -> modelo optimizado -> test
def run_finetune_pair(name, epochs_final=3, n_bats=5, n_iters=3, epochs_search=1,
                      search_train=1500, search_val=200, seed=42, save_model=True):
    out = Path("salida") / name / "finetune"
    out.mkdir(parents=True, exist_ok=True)
    print(f"\n{'=' * 70}\n  FINE-TUNING ES -> {name.upper()}  ({PRETRAINED})\n{'=' * 70}")

    _, train, val, test = prepare_data(name)          # mismos datos que el modelo desde cero
    tr = [(d[2], d[3]) for d in train]
    va = [(d[2], d[3]) for d in val]
    te = [(d[2], d[3]) for d in test]
    tok = make_tok(name)
    res = {"n_train": len(tr), "n_val": len(va), "n_test": len(te), "modelo": PRETRAINED,
           "etiqueta_destino": TGT_CODE[name]}

    # 0) referencia sin fine-tuning (solo tiene sentido si el idioma está en el modelo)
    if name == "sv":
        m0 = load_pretrained(0.1)
        res["zero_shot"], _ = evaluate_ft(m0, tok, name, te, EVAL_BEAMS)
        z = res["zero_shot"]
        print(f"  ZERO-SHOT test: P={z['precision']:.4f} R={z['recall']:.4f} F1={z['f1']:.4f} BLEU={z['bleu']:.2f}")
        del m0
        free_gpu()

    # 1) fine-tuning con la configuración base
    print("\n  >> Fine-tuning BASE", FT_BASELINE)
    t0 = time.time()
    model, vl, hist = finetune(FT_BASELINE, tr, va, tok, epochs_final, seed, verbose=True)
    m_base, hyp_base = evaluate_ft(model, tok, name, te, EVAL_BEAMS)
    res["ft_base"] = {"cfg": FT_BASELINE, "val_loss": vl, "test": m_base, "historia": hist,
                      "segundos": round(time.time() - t0, 1)}
    print(f"  FT-BASE test: P={m_base['precision']:.4f} R={m_base['recall']:.4f} "
          f"F1={m_base['f1']:.4f} BLEU={m_base['bleu']:.2f}")
    del model
    free_gpu()

    # 2) Bat Algorithm sobre los hiperparámetros del fine-tuning
    rs = random.Random(seed)
    s_tr = rs.sample(tr, min(search_train, len(tr)))
    s_va = rs.sample(va, min(search_val, len(va)))
    cache, trials = {}, []

    def fitness(cfg):
        key = tuple(sorted(cfg.items()))
        if key in cache:
            return cache[key]
        t1, model_, f1 = time.time(), None, 0.0
        try:
            model_, vl_, _ = finetune(cfg, s_tr, s_va, tok, epochs_search, seed)
            if math.isfinite(vl_):
                f1 = evaluate_ft(model_, tok, name, s_va, num_beams=1)[0]["f1"]
        except RuntimeError as e:      # p. ej. memoria insuficiente en la GPU
            print("    (config descartada:", str(e)[:70], ")")
        finally:
            del model_
            free_gpu()
        cache[key] = f1
        trials.append({"cfg": cfg, "f1_val": f1})
        print(f"    prueba #{len(trials):02d} F1(val)={f1:.4f}  {cfg}  ({time.time() - t1:.0f}s)")
        return f1

    print(f"\n  >> Bat Algorithm: {n_bats} murciélagos x {n_iters} iteraciones")
    t0 = time.time()
    best_cfg, best_f, bat_hist = bat_generic(fitness, FT_SPACE, n_bats, n_iters, seed,
                                             init_vectors=[encode_space(FT_BASELINE, FT_SPACE)])
    print("  Mejor configuración:", best_cfg)

    # 3) fine-tuning final con la mejor configuración (mismos datos y épocas que el base)
    print("\n  >> Fine-tuning con la configuración optimizada")
    model, vl, hist = finetune(best_cfg, tr, va, tok, epochs_final, seed, verbose=True)
    m_bat, hyp_bat = evaluate_ft(model, tok, name, te, EVAL_BEAMS)
    res["ft_bat"] = {"cfg": best_cfg, "fitness_busqueda": best_f, "val_loss": vl, "test": m_bat,
                     "historia_entrenamiento": hist, "historia_bat": bat_hist, "pruebas": trials,
                     "segundos": round(time.time() - t0, 1)}
    print(f"  FT-BAT  test: P={m_bat['precision']:.4f} R={m_bat['recall']:.4f} "
          f"F1={m_bat['f1']:.4f} BLEU={m_bat['bleu']:.2f}")
    if save_model:
        model.save_pretrained(out / "modelo_ft_bat")
        tok.save_pretrained(out / "modelo_ft_bat")
    del model
    free_gpu()

    with open(out / "resultados_finetune.json", "w", encoding="utf-8") as f:
        json.dump(res, f, ensure_ascii=False, indent=2)
    with open(out / "ejemplos_test.txt", "w", encoding="utf-8") as f:
        for i, p in enumerate(te[:30]):
            f.write(f"ES  : {p[0]}\nREF : {p[1]}\nBASE: {hyp_base[i]}\nBAT : {hyp_bat[i]}\n\n")
    return res

### Ejecución de la Parte B

Valores por defecto: 5 murciélagos × 3 iteraciones (unas 20 pruebas), cada una con 1 época sobre 1,500 pares y evaluación en 200 de validación. Si la GPU es lenta, baja `search_train`, `n_bats` o `n_iters`. Para sueco agrega `"sv"` a `PARES_B` (requiere la descarga de Europarl, que no se ha probado en tu entorno).

In [ ]:
PARES_B = ["nah"]          # puedes usar ["nah", "sv"]
resultados_b = {}
for par in PARES_B:
    resultados_b[par] = run_finetune_pair(par, epochs_final=3, n_bats=5, n_iters=3,
                                          epochs_search=1, search_train=1500, search_val=200)


  FINE-TUNING ES -> NAH  (facebook/nllb-200-distilled-600M)
[nah] limpieza: {'crudo': 16145, 'sin_nulos_ni_vacios': 16062, 'sin_pares_duplicados': 15448, 'sin_fuentes_duplicadas': 15191, 'sin_copias_ni_sin_letras': 15113, 'tras_filtro_longitud': 13069}
[nah] train=9225 val=1146 test=1171

  >> Fine-tuning BASE {'lr': 5e-05, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'label_smoothing': 0.1, 'dropout': 0.1, 'batch_size': 16, 'freeze_embeddings': True}


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

    época 01  train 5.588  val 3.579
    época 02  train 4.677  val 3.242


[transformers] Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    época 03  train 4.435  val 3.149


[transformers] Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

... (58 outputs left out)

    época 03  train 6.070  val 3.909


[transformers] Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

  FT-BAT  test: P=0.1856 R=0.2729 F1=0.2210 BLEU=4.66


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Resumen final: todos los modelos evaluados en el MISMO conjunto de test ===
def fila(lab, m):
    print(f"{lab:<30}{m['precision']:>10.4f}{m['recall']:>9.4f}{m['f1']:>9.4f}{m['bleu']:>8.2f}")

print(f"{'ES -> NAH':<30}{'Precisión':>10}{'Recall':>9}{'F1':>9}{'BLEU':>8}")
fila("Scratch base", BASE_NAH)
fila("Scratch + BAT", m_bat_scratch)
if "nah" in resultados_b:
    fila("Fine-tuning base", resultados_b["nah"]["ft_base"]["test"])
    fila("Fine-tuning + BAT", resultados_b["nah"]["ft_bat"]["test"])
if "sv" in resultados_b:
    print(f"\n{'ES -> SV':<30}{'Precisión':>10}{'Recall':>9}{'F1':>9}{'BLEU':>8}")
    fila("Zero-shot (sin ajuste)", resultados_b["sv"]["zero_shot"])
    fila("Fine-tuning base", resultados_b["sv"]["ft_base"]["test"])
    fila("Fine-tuning + BAT", resultados_b["sv"]["ft_bat"]["test"])

ES -> NAH                      Precisión   Recall       F1    BLEU
Scratch base                      0.2926   0.3211   0.3062    7.43
Scratch + BAT                     0.3404   0.3488   0.3445    9.57
Fine-tuning base                  0.2899   0.3502   0.3172    8.29
Fine-tuning + BAT                 0.1856   0.2729   0.2210    4.66
